# الفصل 54: أساسيات تقديم النماذج للاستدلال

> هناك فرق هائل بين "تشغيل نموذج" بـ model.generate() في سكربت تجريبي، وبين تقديم آلاف الطلبات المتزامنة بزمن استجابة منخفض وتكلفة محسوبة بدقة.

- **الجزء:** 11 — النشر والتشغيل
- **الفصل:** 54
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 53: تطبيقات متخصصة حسب المجال
- **الوسوم:** Serving، vLLM، TGI، KV Cache

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch54-model-serving.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-11-llmops%2Fch54-model-serving.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-11-llmops/ch54-model-serving.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch54-model-serving.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch54-model-serving.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch54-model-serving.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** requests, torch, vllm
- **تثبيت يدوي:** vllm — ثقيلة أو تحتاج إعدادًا خاصًا. شغّل `%pip install vllm` قبل خلاياها.
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("requests", "requests"),
    ("torch", "torch"),
    ("vllm", "vllm")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch54-model-serving.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-11-llmops%2Fch54-model-serving.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-11-llmops/ch54-model-serving.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch54-model-serving.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch54-model-serving.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch54-model-serving.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> هناك فرق هائل بين "تشغيل نموذج" بـ model.generate() في سكربت تجريبي، وبين تقديم آلاف الطلبات المتزامنة بزمن استجابة منخفض وتكلفة محسوبة بدقة.

> **بيانات الفصل**
>
> - **الجزء:** 11 — النشر والتشغيل
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 53: تطبيقات متخصصة حسب المجال


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم لماذا الاستدلال التسلسلي بطيء جوهريًا؟
- فهم KV Cache — التحسين الأساسي الذي يجعل الاستدلال ممكنًا عمليًا
- فهم **حساب حجم KV Cache** — معرفة هذا الحساب ضرورية لتخطيط سعة الخادم


## 1. لماذا يهمّك هذا الموضوع؟

هناك فرق هائل بين "تشغيل نموذج" بـ `model.generate()` في سكربت تجريبي، وبين تقديم آلاف الطلبات المتزامنة بزمن استجابة منخفض وتكلفة محسوبة بدقة. هذا الفصل يبني الأساس الذي يقوم عليه كل نظام إنتاجي حقيقي.


## 2. الخريطة الذهنية


```text
تحديات تقديم LLM للاستدلال
    │
    ├── الاستهلاك الهائل للذاكرة (أوزان + KV Cache)
    ├── طبيعة التوليد التسلسلية (token واحد في كل مرة)
    ├── تفاوت أطوال الطلبات (بعضها قصير، بعضها طويل جدًا)
    └── الحاجة لخدمة طلبات متعددة متزامنة بكفاءة
```


## 3. الشرح المتدرج

### 🟢 أ) لماذا الاستدلال التسلسلي بطيء جوهريًا؟

التوليد التنبؤي التالي (Autoregressive) يعني أن كل token جديد يتطلب تمريرة أمامية كاملة عبر النموذج، معتمدًا على كل الـ tokens السابقة:


```text
توليد "أحب القراءة كثيرًا" يتطلب 4 تمريرات أمامية متتالية:
الخطوة 1: [أحب]                  -> يُنتج "القراءة"
الخطوة 2: [أحب، القراءة]          -> يُنتج "كثيرًا"
الخطوة 3: [أحب، القراءة، كثيرًا]  -> يُنتج [EOS]
```


🟢 **الحدس**: هذا يعني أن توليد جملة طويلة مكلف خطيًا بعدد الـ tokens، وعلى عكس التدريب (حيث نُعالج كل التسلسل دفعة واحدة بالتوازي)، الاستدلال **تسلسلي بطبيعته** — وهذا هو مصدر أغلب تحديات الأداء.

### 🟡 ب) KV Cache — التحسين الأساسي الذي يجعل الاستدلال ممكنًا عمليًا

بدون KV Cache، كل token جديد يتطلب إعادة حساب Attention لكل الـ tokens السابقة من الصفر — تكرار هائل وغير ضروري.


In [ ]:
import torch
import torch.nn.functional as F

class AttentionWithKVCache:
    """
    توضيح مفاهيمي لآلية KV Cache:
    نحتفظ بـ K وV المحسوبة سابقًا، ونضيف فقط K,V الخاصة بالـ token الجديد
    """
    def __init__(self, d_model, n_heads):
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = torch.nn.Linear(d_model, d_model)
        self.k_proj = torch.nn.Linear(d_model, d_model)
        self.v_proj = torch.nn.Linear(d_model, d_model)
        self.o_proj = torch.nn.Linear(d_model, d_model)
        self.cache_k, self.cache_v = None, None   # الذاكرة المخبّأة

    def forward_with_cache(self, x_new_token: torch.Tensor) -> torch.Tensor:
        """x_new_token: [batch, 1, d_model] — رمز واحد جديد فقط!"""
        batch_size = x_new_token.shape[0]

        q = self.q_proj(x_new_token).view(batch_size, 1, self.n_heads, self.head_dim)
        k_new = self.k_proj(x_new_token).view(batch_size, 1, self.n_heads, self.head_dim)
        v_new = self.v_proj(x_new_token).view(batch_size, 1, self.n_heads, self.head_dim)

        if self.cache_k is None:
            self.cache_k, self.cache_v = k_new, v_new
        else:
            # الإضافة فقط - وليس إعادة الحساب الكامل!
            self.cache_k = torch.cat([self.cache_k, k_new], dim=1)
            self.cache_v = torch.cat([self.cache_v, v_new], dim=1)

        # الاستعلام الجديد ينتبه لكل التاريخ المخزّن (القديم + الجديد)
        attn_scores = torch.einsum("bqhd,bkhd->bhqk", q, self.cache_k) / (self.head_dim ** 0.5)
        attn_weights = F.softmax(attn_scores, dim=-1)
        output = torch.einsum("bhqk,bkhd->bqhd", attn_weights, self.cache_v)

        return self.o_proj(output.reshape(batch_size, 1, self.d_model))

    def reset_cache(self):
        self.cache_k, self.cache_v = None, None


### 💡 **حساب حجم KV Cache** — معرفة هذا الحساب ضرورية لتخطيط سعة الخادم:

$$
\text{حجم الذاكرة} = 2 \times n_{\text{layers}} \times n_{\text{heads\_kv}} \times d_{\text{head}} \times \text{seq\_len} \times \text{batch\_size} \times \text{bytes\_per\_param}
$$


In [ ]:
def calculate_kv_cache_memory_gb(n_layers, n_kv_heads, head_dim, seq_len, batch_size,
                                    dtype_bytes=2) -> float:
    """
    حساب حجم KV Cache بالجيجابايت — ضروري لتخطيط سعة GPU
    العامل 2 لأننا نخزّن K و V منفصلين
    """
    total_bytes = 2 * n_layers * n_kv_heads * head_dim * seq_len * batch_size * dtype_bytes
    return total_bytes / (1024 ** 3)

# مثال: LLaMA-2-7B مع GQA (8 KV heads)، سياق 4096، batch=16
memory_gb = calculate_kv_cache_memory_gb(
    n_layers=32, n_kv_heads=8, head_dim=128, seq_len=4096, batch_size=16
)
print(f"حجم KV Cache المطلوب: {memory_gb:.2f} GB")  # يوضح لماذا GQA (الفصل 14) ضروري جدًا للإنتاج


> ⚠️ **خطأ شائع**: إهمال حساب KV Cache عند تخطيط سعة الخادم، والتخطيط فقط لحجم أوزان النموذج. في سياقات طويلة ودفعات (batches) كبيرة، **KV Cache قد يتجاوز حجم أوزان النموذج نفسها** بسهولة! هذا بالضبط سبب أهمية GQA/MQA (الفصل 14) في النماذج الحديثة — تقليل عدد رؤوس KV يقلّل حجم الـ cache بشكل مباشر ومتناسب.

### 🔴 ج) طبقة التعمّق: محركات الاستدلال الإنتاجية (vLLM وTGI)

كتابة خادم استدلال من الصفر غير عملي في الإنتاج — المحركات المتخصصة تُطبّق تحسينات متقدمة جدًا (PagedAttention، Continuous Batching، سنتعمّق فيها بالفصل القادم).


In [ ]:
# ==================== استخدام vLLM (أشهر محرك استدلال مفتوح المصدر) ====================
from vllm import LLM, SamplingParams

def setup_vllm_server(model_path: str, tensor_parallel_size=1):
    """
    vLLM يُطبّق تلقائيًا: PagedAttention، Continuous Batching،
    وتحسينات ذاكرة متقدمة تزيد throughput بعشرات الأضعاف مقارنة بـ HuggingFace العادي
    """
    llm = LLM(
        model=model_path,
        tensor_parallel_size=tensor_parallel_size,   # توزيع عبر عدة GPUs
        gpu_memory_utilization=0.9,                   # استغلال 90% من ذاكرة GPU لـ KV cache
        max_model_len=8192,
        dtype="bfloat16",
        enable_prefix_caching=True,                   # تخزين مؤقت لبادئات مشتركة (system prompts)
    )
    return llm

def batch_inference_with_vllm(llm, prompts: list) -> list:
    sampling_params = SamplingParams(
        temperature=0.7, top_p=0.9, max_tokens=512,
        stop=["<|end|>"]
    )
    # vLLM يُجدول كل الطلبات معًا بكفاءة عبر Continuous Batching تلقائيًا
    outputs = llm.generate(prompts, sampling_params)
    return [output.outputs[0].text for output in outputs]

# ==================== بديل: Text Generation Inference (TGI) من Hugging Face ====================
"""
تشغيل TGI عبر Docker (النهج الإنتاجي الأكثر شيوعًا):

docker run --gpus all --shm-size 1g -p 8080:80 \
    -v $PWD/models:/data \
    ghcr.io/huggingface/text-generation-inference:latest \
    --model-id meta-llama/Llama-2-7b-chat-hf \
    --max-total-tokens 4096 \
    --max-batch-prefill-tokens 8192 \
    --quantize bitsandbytes-nf4
"""

import requests

def query_tgi_server(prompt: str, server_url="http://localhost:8080") -> str:
    response = requests.post(
        f"{server_url}/generate",
        json={
            "inputs": prompt,
            "parameters": {"max_new_tokens": 256, "temperature": 0.7, "top_p": 0.9}
        }
    )
    return response.json()["generated_text"]


## 4. مثال محلول خطوة بخطوة

**المعطيات:** Llama-3-8B، عدد الطبقات 32، عدد رؤوس KV = 8 (GQA)، بُعد الرأس 128، سياق 8192، دفعة 32، دقة FP16 (بايتان لكل قيمة).

**حساب KV Cache لطلب واحد:**

\[
2 \times 32 \times 8 \times 128 \times 8192 \times 2 = 1{,}073{,}741{,}824 \text{ بايت} \approx 1 \text{ GiB}
\]

**لدفعة من 32 طلبًا:** \(32 \times 1 = 32\) GiB.

**مقارنة مع MHA:** لو كان عدد رؤوس KV = 32 بدلًا من 8، لكان الحجم \(4 \times 32 = 128\) GiB، وهو أكبر من ذاكرة A100 (80GB) قبل حساب الأوزان. هنا يتضح لماذا GQA ضروري للإنتاج.

**تخطيط السعة على A100 80GB:**

| العنصر | الحجم |
|---|---:|
| أوزان النموذج (FP16) | 16 GiB |
| KV Cache لدفعة 32 | 32 GiB |
| مساحة عمل/تجزئة | ~8 GiB |
| الإجمالي | ~56 GiB |
| المتبقي | ~24 GiB |

يمكن زيادة الدفعة إلى نحو 50-55 طلبًا قبل الوصول إلى الحد، مع مراقبة التجزئة. لاحظ أن السياق الطويل هو العامل الأكثر تأثيرًا: مضاعفة السياق تضاعف KV Cache مباشرة.

**الخلاصة:** لا تخطّط لحجم الأوزان فقط. KV Cache قد يتجاوز الأوزان، ويحدّد عدد الطلبات المتزامنة وسرعة الخدمة.


## 5. نصائح وحيل

> 💡 **قِس زمنين منفصلين**: TTFT (زمن أول token) وTPOT (زمن كل token لاحق). الأول يعكس Prefill والثاني يعكس Decode؛ تحسين كل منهما مختلف.

> 💡 **حدّد `max_tokens` واقعيًا**: كل token إضافي يستهلك وقتًا وذاكرة. لا تسمح بتوليد 4096 token لمهمة تحتاج 200.

> 💡 **فعّل Prefix Caching**: إذا كان system prompt ثابتًا، فإعادة استخدام KV الخاص به توفّر Prefill متكررًا.

> 💡 **راقب ذاكرة GPU لا الاستخدام فقط**: `nvidia-smi` قد يُظهر 90% استخدامًا بينما الذاكرة ممتلئة والتجزئة تسبب OOM. راقب `gpu_memory_used` و`kv_cache_usage`.

> 💡 **اختبر التكميم على مهمتك**: FP8/INT8 قد يوفّر ذاكرة وسرعة، لكن الجودة تختلف حسب النموذج والمهمة. قِس قبل النشر.


## 6. أخطاء شائعة ومصائد

> ⚠️ **إهمال KV Cache في تخطيط السعة**: أشهر خطأ. النموذج يعمل في التجربة بطلب واحد، ثم يفشل في الإنتاج بسبب 32 طلبًا متزامنًا.

> ⚠️ **استخدام `model.generate()` في الإنتاج**: لا يستفيد من continuous batching ولا PagedAttention، ويُهدر الذاكرة والزمن.

> ⚠️ **سياق أقصى مبالغ فيه**: ضبط `max_model_len=128k` لكل الطلبات يحجز ذاكرة ضخمة ويقلّل الدفعة. اضبطه على احتياجك الفعلي.

> ⚠️ **إغفال زمن التحميل (Cold Start)**: تحميل نموذج 8B قد يستغرق دقائق. استخدم readiness probe مناسبة، واحتفظ بنسخ دافئة.

> ⚠️ **مقارنة المحركات دون قياس**: أرقام throughput المنشورة تختلف حسب العتاد والطول والدقة. قِس على حملك الحقيقي.


## 7. جدول مقارنة

| المحرك | الجهة | أبرز ميزاته | الاستخدام المناسب |
|---|---|---|---|
| **vLLM** | UC Berkeley | PagedAttention، أسرع throughput مفتوح المصدر | الخيار الافتراضي لمعظم عمليات النشر الحديثة |
| **TGI** | Hugging Face | تكامل سلس مع Hub، دعم واسع للنماذج | بيئات Hugging Face، سهولة الإعداد |
| **TensorRT-LLM** | NVIDIA | أعلى أداء خام على عتاد NVIDIA | أقصى أداء ممكن، استثمار هندسي أكبر |
| **llama.cpp** | مجتمع مفتوح | استدلال فعّال على CPU/أجهزة محدودة | تطبيقات edge، أجهزة شخصية |
| **SGLang** | LMSYS | محرك حديث بأداء عالٍ، دعم Structured output متقدم | تطبيقات تحتاج توليد مُهيكَل معقّد |


## 8. ملخص أفضل الممارسات

- [ ] احسب حجم KV Cache المتوقع بناءً على أقصى سياق ودفعة متوقعين
- [ ] استخدم GQA/MQA دائمًا إذا كانت المعمارية تدعمها لتقليل ضغط الذاكرة
- [ ] استخدم محرك استدلال متخصص (vLLM/TGI) بدلاً من `generate()` الأساسي في الإنتاج دائمًا
- [ ] فعّل Prefix Caching إذا كانت الطلبات تشترك في system prompt ثابت
- [ ] اختبر الكمّنة (Quantization) للاستدلال (nf4/int8/fp8) وقِس الأثر على الجودة والسرعة

---


## 9. أسئلة مقابلات

**سؤال 1: لماذا يكون توليد LLM بطيئًا بطبيعته؟**

> **إجابة نموذجية:** لأن التوليد autoregressive: كل token يعتمد على كل ما سبقه، فيتطلب تمريرة أمامية كاملة عبر النموذج. لا يمكن توليد 100 token دفعة واحدة كما نعالج تسلسل التدريب، لأن كل token جديد يحتاج ناتج الذي قبله. هذا يجعل زمن التوليد خطيًا بعدد الرموز، ويجعل الذاكرة (KV Cache) تنمو مع طول السياق.

**سؤال 2: اشرح معادلة KV Cache وعواملها.**

> **إجابة نموذجية:** الحجم = \(2 \times L \times H_{kv} \times d_{head} \times S \times B \times \text{bytes}\). العامل 2 لأننا نخزّن K وV. \(L\) عدد الطبقات، \(H_{kv}\) عدد رؤوس KV (وليس رؤوس الانتباه كلها عند GQA)، \(d_{head}\) بُعد الرأس، \(S\) طول التسلسل، \(B\) الدفعة. زيادة السياق أو الدفعة تزيد الذاكرة خطيًا، ولهذا تحتاج GQA/MQA لتقليل \(H_{kv}\).

**سؤال 3: متى تختار vLLM مقابل TGI؟**

> **إجابة نموذجية:** vLLM خيار افتراضي قوي بسبب PagedAttention وcontinuous batching وأداء throughput عالٍ. TGI مناسب إذا كنت داخل منظومة Hugging Face وتريد تكاملًا سهلًا ودعمًا واسعًا للنماذج. القرار النهائي يعتمد على النموذج المدعوم، العتاد، والميزات المطلوبة (مثل structured output أو tensor parallelism). قِس على حملك قبل الاختيار.


## 10. تمرين عملي

**الهدف:** قياس أداء خدمة استدلال صغيرة ومقارنة إعدادين.

**الخطوات:**
1. شغّل نموذجًا صغيرًا (مثل Qwen2.5-1.5B أو Llama-3.2-1B) عبر vLLM.
2. أرسل 100 طلب متزامن بأطوال خرج 64 و256 token.
3. سجّل: throughput (token/s)، TTFT p50/p95، TPOT، وذاكرة GPU.
4. أعد التجربة مع `max_model_len` أصغر و`gpu_memory_utilization` أعلى.
5. أعد التجربة مع تعطيل prefix caching (إن أمكن) وقارن.
6. احسب التكلفة لكل مليون token بناءً على سعر GPU/الساعة.

**معيار النجاح:** تقرير يوضح الإعداد الأفضل لحملك، مع تفسير الفرق وليس مجرد أرقام.


## 11. مراجع للتعمّق

- Kwon, W., et al. (2023). *Efficient Memory Management for Large Language Model Serving with PagedAttention*. SOSP.
- Hugging Face. *Text Generation Inference Documentation*. huggingface.co/docs/text-generation-inference.
- vLLM Documentation. docs.vllm.ai.
- Dao, T., et al. (2022). *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*. NeurIPS.
- Pope, R., et al. (2023). *Efficiently Scaling Transformer Inference*. MLSys.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — النشر والتشغيل / الفصل 54: أساسيات تقديم النماذج للاستدلال.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
